# LTIC on Pl@ntNet-300K (Kaggle)

Before running, set these in the notebook's right-hand panel (full guide: `KAGGLE.md` in the repo):

1. **Accelerator:** `GPU T4 x2`
2. **Internet:** On
3. **Input:** add the dataset **plantnet 300k images** (`noahbadoa/plantnet-300k-images`)

Then click **Save Version -> Save & Run All (Commit)**. The run continues even if you close the browser.

In [ ]:
# ---- Settings: the only cell you need to edit ----
GITHUB_USER = "Raiyan007-gb"       # GitHub account you pushed the branch to (e.g. "moon-moon708" for your fork)
BRANCH      = "plantnet-bioclip2"
BACKBONE    = "bioclip2"           # bioclip2 | dinov2_l14 | clip_b32
EPOCHS      = 200
CACHE_VIEWS = 5                    # augmented copies of each training image whose features are pre-computed
BATCH       = 256                  # lower to 128 if you get "CUDA out of memory"

## Step 1: check the GPU and the dataset

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input

## Step 2: get the code from GitHub

In [ ]:
import os
CODE = "/kaggle/working/LTIC-Herb"
REPO_URL = f"https://github.com/{GITHUB_USER}/LTIC-Herb.git"
if os.path.isdir(CODE):
    !git -C {CODE} pull
else:
    !git clone -b {BRANCH} {REPO_URL} {CODE}
!git -C {CODE} log --oneline -1

## Step 3: prepare the data, train, and evaluate on the test split

Runs `kaggle/run_kaggle.sh`: installs open_clip, writes the split files, extracts the encoder features once,
trains the LTIC head for `EPOCHS` epochs, then evaluates the best checkpoint on the official test split.

In [ ]:
os.environ.update(BACKBONE=BACKBONE, EPOCHS=str(EPOCHS), CACHE_VIEWS=str(CACHE_VIEWS), BATCH=str(BATCH))
!bash {CODE}/kaggle/run_kaggle.sh
if get_ipython().user_ns.get("_exit_code", 0) != 0:
    raise RuntimeError("The run stopped with an error - scroll up to the first error message.")

## Results

Saved in the notebook **Output** (`/kaggle/working/runs/plantnetDataset/`):

- `plantnet_<backbone>_bt<BATCH>/train.log`: training log, one ` * Acc@1 ...` summary line per epoch (validation set)
- `plantnet_<backbone>_bt<BATCH>/model_best.pth.tar`: best checkpoint (LTIC head only)
- `plantnet_<backbone>_bt<BATCH>_test/train.log`: final **test** result (last ` * Acc@1 ...` line)

`Acc@1`/`Acc@5` are percentages; `HAcc`/`MAcc`/`TAcc` (many/medium/few-shot) and `MacroAcc` are fractions between 0 and 1.

In [ ]:
!find /kaggle/working/runs -name "train.log" -o -name "model_best.pth.tar"